## Overview

Agent Development Kit (ADK in short) is a flexible and modular open source framework for developing and deploying AI agents. While ADK has its own evaluation module, using Vertex AI Gen AI Evaluation provides a toolkit of quality controlled and explainable methods and metrics to evaluate any generative model or application, including agents, and benchmark the evaluation results against your own judgment, using your own evaluation criteria.

This tutorial shows how to evaluate an ADK agent using Vertex AI Gen AI Evaluation for agent evaluation.

The steps performed include:

* Build local agent using ADK
* Prepare Agent Evaluation dataset
* Single tool usage evaluation
* Trajectory evaluation
* Response evaluation

## Get started

### Set Google Cloud project information

In [1]:
import os
from dotenv import load_dotenv
# Define the path to the .env file relative to the notebooks directory
# If you running the notebook from the 'notebooks' directory:
dotenv_path = os.path.join(os.getcwd(), "..", "app", ".env")
# If you launched Jupyter from the project root directory, use this instead:
# dotenv_path = os.path.join(os.getcwd(), "app", ".env")
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path)
    print(f"Loaded environment variables from {dotenv_path}")
else:
    print(f"Could not find .env file at {dotenv_path}")

PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
if PROJECT_ID is None:
    raise ValueError('PROJECT ID NOT SET AND NOT CORRECTLY RETRIEVED. FIX BEFORE CONTINUING')

BUCKET_URI = f"gs://{PROJECT_ID}-optimization/evaluation/"

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "europe-west1")
print("PROJECT ID => ", PROJECT_ID)
print("BUCKET URI => ", BUCKET_URI)

import vertexai

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

EXPERIMENT_NAME = "evaluate-adk-agent"  # @param {type:"string"}

vertexai.init(project=PROJECT_ID, location=LOCATION, experiment=EXPERIMENT_NAME)

Loaded environment variables from /Users/julienrzeznik/projects/AGENTOPS/amadeus/workshop/immersion-day-template/notebooks/../app/.env
PROJECT ID =>  immersion-day-sandbox-008
BUCKET URI =>  gs://immersion-day-sandbox-008-optimization/evaluation/


I0403 02:47:59.660611 2302644 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)
I0403 02:47:59.715141 2302673 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)


## Import libraries

Import tutorial libraries.

In [2]:
import json
import asyncio

# General
import random
import string
from typing import Any

from IPython.display import HTML, Markdown, display
from google.adk.agents import Agent

# Build agent with adk
from google.adk.events import Event
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# Evaluate agent
from google.cloud import aiplatform
from google.genai import types
import pandas as pd
import plotly.graph_objects as go
from vertexai.preview.evaluation import EvalTask
from vertexai.preview.evaluation.metrics import (
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
    TrajectorySingleToolUse,
)

## Define helper functions

Initiate a set of helper functions to print tutorial results.

In [3]:
def get_id(length: int = 8) -> str:
    """Generate a uuid of a specified length (default=8)."""
    return "".join(random.choices(string.ascii_lowercase + string.digits, k=length))


def parse_adk_output_to_dictionary(events: list[Event], *, as_json: bool = False):
    """
    Parse ADK event output into a structured dictionary format,
    with the predicted trajectory dumped as a JSON string.

    """

    final_response = ""
    trajectory = []

    for event in events:
        if not getattr(event, "content", None) or not getattr(event.content, "parts", None):
            continue
        for part in event.content.parts:
            if getattr(part, "function_call", None):
                info = {
                    "tool_name": part.function_call.name,
                    "tool_input": dict(part.function_call.args),
                }
                if info not in trajectory:
                    trajectory.append(info)
            if event.content.role == "model" and getattr(part, "text", None):
                final_response = part.text.strip()

    if as_json:
        trajectory_out = json.dumps(trajectory)
    else:
        trajectory_out = trajectory

    return {"response": final_response, "predicted_trajectory": trajectory_out}


def format_output_as_markdown(output: dict) -> str:
    """Convert the output dictionary to a formatted markdown string."""
    markdown = "### AI Response\n" + output["response"] + "\n\n"
    if output["predicted_trajectory"]:
        markdown += "### Function Calls\n"
        for call in output["predicted_trajectory"]:
            markdown += f"- **Function**: `{call['tool_name']}`\n"
            markdown += "  - **Arguments**\n"
            for key, value in call["tool_input"].items():
                markdown += f"    - `{key}`: `{value}`\n"
    return markdown


def display_eval_report(eval_result: pd.DataFrame) -> None:
    """Display the evaluation results."""
    display(Markdown("### Summary Metrics"))
    display(
        pd.DataFrame(
            eval_result.summary_metrics.items(), columns=["metric", "value"]
        )
    )
    if getattr(eval_result, "metrics_table", None) is not None:
        display(Markdown("### Row‑wise Metrics"))
        display(eval_result.metrics_table.head())


def display_drilldown(row: pd.Series) -> None:
    """Displays a drill-down view for trajectory data within a row."""

    style = "white-space: pre-wrap; width: 800px; overflow-x: auto;"

    if not (
        isinstance(row["predicted_trajectory"], list)
        and isinstance(row["reference_trajectory"], list)
    ):
        return

    for predicted_trajectory, reference_trajectory in zip(
        row["predicted_trajectory"], row["reference_trajectory"]
    ):
        display(
            HTML(
                f"<h3>Tool Names:</h3><div style='{style}'>{predicted_trajectory['tool_name'], reference_trajectory['tool_name']}</div>"
            )
        )

        if not (
            isinstance(predicted_trajectory.get("tool_input"), dict)
            and isinstance(reference_trajectory.get("tool_input"), dict)
        ):
            continue

        for tool_input_key in predicted_trajectory["tool_input"]:
            print("Tool Input Key: ", tool_input_key, flush=True)

            if tool_input_key in reference_trajectory["tool_input"]:
                print(
                    "Tool Values: ",
                    predicted_trajectory["tool_input"][tool_input_key],
                    reference_trajectory["tool_input"][tool_input_key],
                    flush=True,
                )
            else:
                print(
                    "Tool Values: ",
                    predicted_trajectory["tool_input"][tool_input_key],
                    "N/A",
                    flush=True,
                )
        print("\n", flush=True)
    display(HTML("<hr>"))


def display_dataframe_rows(
    df: pd.DataFrame,
    columns: list[str] | None = None,
    num_rows: int = 3,
    display_drilldown: bool = False,
) -> None:
    """Displays a subset of rows from a DataFrame, optionally including a drill-down view."""

    if columns:
        df = df[columns]

    base_style = "font-family: monospace; font-size: 14px; white-space: pre-wrap; width: auto; overflow-x: auto;"
    header_style = base_style + "font-weight: bold;"

    for _, row in df.head(num_rows).iterrows():
        for column in df.columns:
            display(
                HTML(
                    f"<span style='{header_style}'>{column.replace('_', ' ').title()}: </span>"
                )
            )
            display(HTML(f"<span style='{base_style}'>{row[column]}</span><br>"))

        display(HTML("<hr>"))

        if (
            display_drilldown
            and "predicted_trajectory" in df.columns
            and "reference_trajectory" in df.columns
        ):
            display_drilldown(row)


def plot_bar_plot(
    eval_result: pd.DataFrame, title: str, metrics: list[str] = None, render_plotly: bool = False
) -> None:
    """Plot summary metrics as a bar plot. Set render_plotly=True to use interactive Plotly."""
    print("--- plot_bar_plot ---", flush=True)
    
    summary_metrics = eval_result.summary_metrics
    
    if metrics:
        filtered_metrics = {}
        for k, v in summary_metrics.items():
            match_found = False
            for selected_metric in metrics:
                if selected_metric in k or k in selected_metric:
                    match_found = True
                    break
            if match_found:
                filtered_metrics[k] = v
        summary_metrics = filtered_metrics

    # --- Fallback Text-based Chart ---
    print("\n📊 --- Metrics Bar Chart ---", flush=True)
    max_val = max(summary_metrics.values()) if summary_metrics.values() else 1
    for k, v in summary_metrics.items():
        val = 0 if pd.isna(v) else v
        bar_len = int((val / max_val) * 20) if max_val > 0 else 0
        clean_name = k.replace('/mean', '').replace('trajectory_', '')
        print(f"{clean_name:<25} | {'█' * bar_len} ({val:.2f})", flush=True)
    print("----------------------------\n", flush=True)

    if render_plotly:
        data = []
        data.append(
            go.Bar(
                x=list(summary_metrics.keys()),
                y=list(summary_metrics.values()),
                name=title,
            )
        )
        fig = go.Figure(data=data)
        fig.update_layout(barmode="group")
        print("Showing interactive plot...", flush=True)
        try:
            fig.show()
        except Exception as e:
            print(f"Error showing plot: {e}", flush=True)


def display_radar_plot(eval_results, title: str, metrics=None, render_plotly: bool = False):
    """Plot summary metrics as a radar plot. Set render_plotly=True to use interactive Plotly."""
    print("--- display_radar_plot ---", flush=True)
    
    summary_metrics = eval_results.summary_metrics
    if metrics:
        summary_metrics = {
            k: summary_metrics[k]
            for k, v in summary_metrics.items()
            if any(selected_metric in k for selected_metric in metrics)
        }

    # --- Fallback Text-based Chart ---
    print("\n📊 --- Metrics Chart ---", flush=True)
    max_val = max(summary_metrics.values()) if summary_metrics.values() else 1
    for k, v in summary_metrics.items():
        val = 0 if pd.isna(v) else v
        bar_len = int((val / max_val) * 20) if max_val > 0 else 0
        clean_name = k.replace('/mean', '').replace('trajectory_', '')
        print(f"{clean_name:<25} | {'█' * bar_len} ({val:.2f})", flush=True)
    print("------------------------\n", flush=True)

    if render_plotly:
        min_val = min(summary_metrics.values())
        max_val = max(summary_metrics.values())

        fig = go.Figure()
        fig.add_trace(
            go.Scatterpolar(
                r=list(summary_metrics.values()),
                theta=list(summary_metrics.keys()),
                fill="toself",
                name=title,
            )
        )
        fig.update_layout(
            title=title,
            polar=dict(radialaxis=dict(visible=True, range=[min_val, max_val])),
            showlegend=True,
        )
        fig.show()

## Build ADK agent

Build your application using ADK, including the Gemini model and custom tools that you define.

### Set tools

To start, set the tools that a customer support agent needs to do their job.

In [4]:
def get_product_details(product_name: str):
    """Gathers basic details about a product."""
    details = {
        "smartphone": "A cutting-edge smartphone with advanced camera features and lightning-fast processing.",
        "usb charger": "A super fast and light usb charger",
        "shoes": "High-performance running shoes designed for comfort, support, and speed.",
        "headphones": "Wireless headphones with advanced noise cancellation technology for immersive audio.",
        "speaker": "A voice-controlled smart speaker that plays music, sets alarms, and controls smart home devices.",
    }
    return details.get(product_name, "Product details not found.")


def get_product_price(product_name: str):
    """Gathers price about a product."""
    details = {
        "smartphone": 500,
        "usb charger": 10,
        "shoes": 100,
        "headphones": 50,
        "speaker": 80,
    }
    return details.get(product_name, "Product price not found.")

### Set the model

Choose which Gemini AI model your agent will use. If you're curious about Gemini and its different capabilities, take a look at [the official documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models) for more details.

In [5]:
model = "gemini-2.5-flash"

### Assemble the agent

The Vertex AI Gen AI Evaluation works directly with 'Queryable' agents, and also lets you add your own custom functions with a specific structure (signature).

In this case, you assemble the agent using a custom function. The function triggers the agent for a given input and parse the agent outcome to extract the response and called tools.

In [6]:
async def agent_parsed_outcome(query):
   app_name = "product_research_app"
   user_id = "user_one"
   session_id = "session_one"
   
   product_research_agent = Agent(
       name="ProductResearchAgent",
       model=model,
       description="An agent that performs product research.",
       instruction=f"""
       Analyze this user request: '{query}'.
       If the request is about price, use get_product_price tool.
       Otherwise, use get_product_details tool to get product information.
       """,
       tools=[get_product_details, get_product_price],
   )

   session_service = InMemorySessionService()
   await session_service.create_session(
       app_name=app_name, user_id=user_id, session_id=session_id
   )

   runner = Runner(
       agent=product_research_agent, app_name=app_name, session_service=session_service
   )

   content = types.Content(role="user", parts=[types.Part(text=query)])
   events = [event async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content)]
   
   return parse_adk_output_to_dictionary(events)


In [7]:
# --- Sync wrapper for Vertex‑AI evaluation
def agent_parsed_outcome_sync(prompt: str):
    result = asyncio.run(agent_parsed_outcome(prompt))
    result["predicted_trajectory"] = json.dumps(result["predicted_trajectory"])
    return result

### Test the agent

Query your agent.

In [8]:
response = await agent_parsed_outcome(query="Get product details for shoes")
display(Markdown(format_output_as_markdown(response)))

### AI Response
High-performance running shoes designed for comfort, support, and speed.

### Function Calls
- **Function**: `get_product_details`
  - **Arguments**
    - `product_name`: `shoes`


In [9]:
response = await agent_parsed_outcome(query="Get product price for shoes")
display(Markdown(format_output_as_markdown(response)))

### AI Response
The price of shoes is $100.

### Function Calls
- **Function**: `get_product_price`
  - **Arguments**
    - `product_name`: `shoes`


## Evaluating a ADK agent with Vertex AI Gen AI Evaluation

When working with AI agents, it's important to keep track of their performance and how well they're working. You can look at this in two main ways: **monitoring** and **observability**.

Monitoring focuses on how well your agent is performing specific tasks:

* **Single Tool Selection**: Is the agent choosing the right tools for the job?

* **Multiple Tool Selection (or Trajectory)**: Is the agent making logical choices in the order it uses tools?

* **Response generation**: Is the agent's output good, and does it make sense based on the tools it used?

Observability is about understanding the overall health of the agent:

* **Latency**: How long does it take the agent to respond?

* **Failure Rate**: How often does the agent fail to produce a response?

Vertex AI Gen AI Evaluation service helps you to assess all of these aspects both while you are prototyping the agent or after you deploy it in production. It provides [pre-built evaluation criteria and metrics](https://cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval) so you can see exactly how your agents are doing and identify areas for improvement.

### Prepare Agent Evaluation dataset

To evaluate your AI agent using the Vertex AI Gen AI Evaluation service, you need a specific dataset depending on what aspects you want to evaluate of your agent.  

This dataset should include the prompts given to the agent. It can also contain the ideal or expected response (ground truth) and the intended sequence of tool calls the agent should take (reference trajectory) representing the sequence of tools you expect agent calls for each given prompt.

> Optionally, you can provide both generated responses and predicted trajectory (**Bring-Your-Own-Dataset scenario**).

Below you have an example of dataset you might have with a customer support agent with user prompt and the reference trajectory.

In [10]:
eval_data = {
    "prompt": [
        "Get price for smartphone",
        "Get product details and price for headphones",
        "Get details for usb charger",
        "Get product details and price for shoes",
        "Get product details for speaker?",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "smartphone"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "headphones"},
            },
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "headphones"},
            },
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "usb charger"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "shoes"},
            },
            {"tool_name": "get_product_price", "tool_input": {"product_name": "shoes"}},
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "speaker"},
            }
        ],
    ],
}

eval_sample_dataset = pd.DataFrame(eval_data)

Print some samples from the dataset.

In [11]:
display_dataframe_rows(eval_sample_dataset, num_rows=3)

### Single tool usage evaluation

After you've set your AI agent and the evaluation dataset, you start evaluating if the agent is choosing the correct single tool for a given task.


#### Set single tool usage metrics

The `trajectory_single_tool_use` metric in Vertex AI Gen AI Evaluation gives you a quick way to evaluate whether your agent is using the tool you expect it to use, regardless of any specific tool order. It's a basic but useful way to start evaluating if the right tool was used at some point during the agent's process.

To use the `trajectory_single_tool_use` metric, you need to set what tool should have been used for a particular user's request. For example, if a user asks to "send an email", you might expect the agent to use an "send_email" tool, and you'd specify that tool's name when using this metric.


In [12]:
single_tool_usage_metrics = [TrajectorySingleToolUse(tool_name="get_product_price")]

#### Run an evaluation task

To run the evaluation, you initiate an `EvalTask` using the pre-defined dataset (`eval_sample_dataset`) and metrics (`single_tool_usage_metrics` in this case) within an experiment. Then, you run the evaluation using agent_parsed_outcome function and assigns a unique identifier to this specific evaluation run, storing and visualizing the evaluation results.


In [13]:
EXPERIMENT_RUN = f"single-metric-eval-{get_id()}"

single_tool_call_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=single_tool_usage_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/single-metric-eval",
)

single_tool_call_eval_result = single_tool_call_eval_task.evaluate(
    runnable=agent_parsed_outcome_sync, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(single_tool_call_eval_result)

I0403 02:48:15.386753 2303354 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:15.652245 2303385 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:15.652564 2303385 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:15.724400 2303415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:15.724475 2303415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:15.959824 2303505 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:15.959913 2303505 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:16.009357 2303534 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:16.009438 2303534 ev_poll_posix.cc:593] FD from fork parent still in poll li

Associating projects/959676885215/locations/europe-west1/metadataStores/default/contexts/evaluate-adk-agent-single-metric-eval-k7v5xolj to Experiment: evaluate-adk-agent


I0403 02:48:17.341248 2303840 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:17.341418 2303840 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:48:17.341434 2303840 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:17.341447 2303840 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:17.549912 2303869 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)


Logging Eval experiment evaluation metadata: {'output_file': 'gs://immersion-day-sandbox-008-optimization/evaluation//single-metric-eval/eval_results_2026-04-03-02-48-14-a0dbb.csv'}


I0403 02:48:17.773145 2303931 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:17.773313 2303931 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:48:17.773317 2303931 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:17.773319 2303931 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:18.005990 2303957 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:18.006094 2303957 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:48:18.006097 2303957 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:18.006098 2303957 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:18.067606 2303985 ev_poll_posix.cc:593] FD from fork parent still in poll li

...


I0403 02:48:21.390603 2305036 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:21.655599 2305068 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:21.655791 2305068 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:48:21.655828 2305068 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:21.655831 2305068 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(91, generation: 1)
I0403 02:48:21.896636 2305114 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:21.896834 2305114 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:48:21.896851 2305114 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
I0403 02:48:21.896882 2305114 ev_poll_posix.cc:593] FD from fork parent still in poll l

### Summary Metrics

,metric,value
0,row_count,5.000000
1,trajectory_single_tool_use/mean,0.600000
2,trajectory_single_tool_use/std,0.547723
3,latency_in_seconds/mean,2.614823
4,latency_in_seconds/std,0.162747
5,failure/mean,0.000000
6,failure/std,0.000000


### Row‑wise Metrics

,prompt,reference_trajectory,response,latency_in_seconds,failure,predicted_trajectory,trajectory_single_tool_use/score
0,Get price for smartphone,"[{'tool_name': 'get_product_price', 'tool_inpu...",The price for the smartphone is 500.,2.60805,0,"[{""tool_name"": ""get_product_price"", ""tool_inpu...",1.0
1,Get product details and price for headphones,"[{'tool_name': 'get_product_details', 'tool_in...",The headphones are wireless with advanced nois...,2.741909,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0
2,Get details for usb charger,"[{'tool_name': 'get_product_details', 'tool_in...",A super fast and light usb charger,2.389737,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",0.0
3,Get product details and price for shoes,"[{'tool_name': 'get_product_details', 'tool_in...",The product details for shoes are: High-perfor...,2.537602,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0
4,Get product details for speaker?,"[{'tool_name': 'get_product_details', 'tool_in...",A voice-controlled smart speaker that plays mu...,2.796818,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",0.0


#### Visualize evaluation results

Use some helper functions to visualize a sample of evaluation result.

In [14]:
display_dataframe_rows(single_tool_call_eval_result.metrics_table, num_rows=3)

### Trajectory Evaluation

After evaluating the agent's ability to select the single most appropriate tool for a given task, you generalize the evaluation by analyzing the tool sequence choices with respect to the user input (trajectory). This assesses whether the agent not only chooses the right tools but also utilizes them in a rational and effective order.

#### Set trajectory metrics

To evaluate agent's trajectory, Vertex AI Gen AI Evaluation provides several ground-truth based metrics:

* `trajectory_exact_match`: identical trajectories (same actions, same order)

* `trajectory_in_order_match`: reference actions present in predicted trajectory, in order (extras allowed)

* `trajectory_any_order_match`: all reference actions present in predicted trajectory (order, extras don't matter).

* `trajectory_precision`: proportion of predicted actions present in reference

* `trajectory_recall`: proportion of reference actions present in predicted.  

All metrics score 0 or 1, except `trajectory_precision` and `trajectory_recall` which range from 0 to 1.

In [15]:
trajectory_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "trajectory_any_order_match",
    "trajectory_precision",
    "trajectory_recall",
]

#### Run an evaluation task

Submit an evaluation by running `evaluate` method of the new `EvalTask`.

In [16]:
EXPERIMENT_RUN = f"trajectory-{get_id()}"

trajectory_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=trajectory_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/multiple-metric-eval",
)

trajectory_eval_result = trajectory_eval_task.evaluate(
    runnable=agent_parsed_outcome_sync, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(trajectory_eval_result)

I0403 02:48:23.709468 2305283 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:23.923209 2305314 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:23.923392 2305314 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:23.981290 2305341 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:23.981349 2305341 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:24.186121 2305372 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:24.186169 2305372 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:24.186171 2305372 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:24.236369 2305400 ev_poll_posix.cc:593] FD from fork parent still in po

Associating projects/959676885215/locations/europe-west1/metadataStores/default/contexts/evaluate-adk-agent-trajectory-dbavqyef to Experiment: evaluate-adk-agent


I0403 02:48:25.717518 2305662 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:25.717666 2305662 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:25.717701 2305662 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:25.717702 2305662 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)


Logging Eval experiment evaluation metadata: {'output_file': 'gs://immersion-day-sandbox-008-optimization/evaluation//multiple-metric-eval/eval_results_2026-04-03-02-48-22-02736.csv'}


I0403 02:48:25.920786 2305753 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:25.920850 2305753 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:25.920851 2305753 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:25.920851 2305753 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:26.118760 2305794 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:26.118904 2305794 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:26.118941 2305794 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:26.118943 2305794 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
  0%|          | 0/5 [00:00<?, ?it/s]I0403 02:48:26.191414 2305862 ev_poll_posix.c

...

I0403 02:48:34.323607 2307231 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:34.323714 2307231 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:34.323730 2307231 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:34.323731 2307231 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:34.504090 2307259 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:34.504215 2307259 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:34.504233 2307259 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:34.504235 2307259 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)


### Summary Metrics

,metric,value
0,row_count,5.000000
1,trajectory_exact_match/mean,0.800000
2,trajectory_exact_match/std,0.447214
3,trajectory_in_order_match/mean,0.800000
4,trajectory_in_order_match/std,0.447214
5,trajectory_any_order_match/mean,1.000000
6,trajectory_any_order_match/std,0.000000
7,trajectory_precision/mean,1.000000
8,trajectory_precision/std,0.000000
9,trajectory_recall/mean,1.000000


### Row‑wise Metrics

,prompt,reference_trajectory,response,latency_in_seconds,failure,predicted_trajectory,trajectory_exact_match/score,trajectory_in_order_match/score,trajectory_any_order_match/score,trajectory_precision/score,trajectory_recall/score
0,Get price for smartphone,"[{'tool_name': 'get_product_price', 'tool_inpu...",The price for the smartphone is 500.,2.902236,0,"[{""tool_name"": ""get_product_price"", ""tool_inpu...",1.0,1.0,1.0,1.0,1.0
1,Get product details and price for headphones,"[{'tool_name': 'get_product_details', 'tool_in...",Wireless headphones with advanced noise cancel...,2.969675,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,1.0,1.0,1.0
2,Get details for usb charger,"[{'tool_name': 'get_product_details', 'tool_in...",A super fast and light usb charger,2.426066,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,1.0,1.0,1.0
3,Get product details and price for shoes,"[{'tool_name': 'get_product_details', 'tool_in...",The price for shoes is 100. The details are: H...,4.778411,0,"[{""tool_name"": ""get_product_price"", ""tool_inpu...",0.0,0.0,1.0,1.0,1.0
4,Get product details for speaker?,"[{'tool_name': 'get_product_details', 'tool_in...",A voice-controlled smart speaker that plays mu...,2.555649,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,1.0,1.0,1.0


#### Visualize evaluation results

Print and visualize a sample of evaluation results.

In [17]:
display_dataframe_rows(trajectory_eval_result.metrics_table, num_rows=3)

In [18]:
plot_bar_plot(
    trajectory_eval_result,
    title="Trajectory Metrics",
    metrics=[f"{metric}/mean" for metric in trajectory_metrics],
)

--- plot_bar_plot ---

📊 --- Metrics Bar Chart ---
exact_match               | ████████████████ (0.80)
in_order_match            | ████████████████ (0.80)
any_order_match           | ████████████████████ (1.00)
precision                 | ████████████████████ (1.00)
recall                    | ████████████████████ (1.00)
----------------------------



### Evaluate final response

Similar to model evaluation, you can evaluate the final response of the agent using Vertex AI Gen AI Evaluation.

#### Set response metrics

After agent inference, Vertex AI Gen AI Evaluation provides several metrics to evaluate generated responses. You can use computation-based metrics to compare the response to a reference (if needed) and using existing or custom model-based metrics to determine the quality of the final response.

Check out the [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval) to learn more.


In [19]:
response_metrics = ["safety", "coherence"]

#### Run an evaluation task

To evaluate agent's generated responses, use the `evaluate` method of the EvalTask class.

In [20]:
EXPERIMENT_RUN = f"response-{get_id()}"

response_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=response_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/response-metric-eval",
)

response_eval_result = response_eval_task.evaluate(
    runnable=agent_parsed_outcome_sync, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_result)

I0403 02:48:43.590304 2307555 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:43.590470 2307555 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:43.590509 2307555 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:43.590511 2307555 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:43.815555 2307589 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:43.815706 2307589 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:43.815730 2307589 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:43.815732 2307589 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)


I0403 02:48:44.013572 2307618 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:44.013828 2307618 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:44.013830 2307618 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:44.013831 2307618 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:44.190632 2307651 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:44.190716 2307651 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:44.190730 2307651 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:44.190730 2307651 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:44.433403 2307704 ev_poll_posix.cc:593] FD from fork parent still in p

Associating projects/959676885215/locations/europe-west1/metadataStores/default/contexts/evaluate-adk-agent-response-6v9vrqoh to Experiment: evaluate-adk-agent


I0403 02:48:46.192295 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)
I0403 02:48:46.192434 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:48:46.192438 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0403 02:48:46.192439 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:48:46.192440 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:48:46.192441 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:48:46.192442 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:48:46.192443 2308070 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(101, generation: 1)
I0403 02:48:46.397075 2308103 ev_poll_posix.cc:593] FD from fork parent still in p

Logging Eval experiment evaluation metadata: {'output_file': 'gs://immersion-day-sandbox-008-optimization/evaluation//response-metric-eval/eval_results_2026-04-03-02-48-43-ca2e8.csv'}


...


I0403 02:49:00.559079 2309555 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)
I0403 02:49:00.771399 2309586 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)


### Summary Metrics

,metric,value
0,row_count,5.000000
1,safety/mean,1.000000
2,safety/std,0.000000
3,coherence/mean,4.200000
4,coherence/std,1.788854
5,latency_in_seconds/mean,2.753157
6,latency_in_seconds/std,0.234121
7,failure/mean,0.000000
8,failure/std,0.000000


### Row‑wise Metrics

,prompt,reference_trajectory,response,latency_in_seconds,failure,predicted_trajectory,safety/explanation,safety/score,coherence/explanation,coherence/score
0,Get price for smartphone,"[{'tool_name': 'get_product_price', 'tool_inpu...",The price for smartphone is 500.,2.787845,0,"[{""tool_name"": ""get_product_price"", ""tool_inpu...",The response is a simple statement providing a...,1.0,The response perfectly and directly answers th...,5.0
1,Get product details and price for headphones,"[{'tool_name': 'get_product_details', 'tool_in...",Wireless headphones with advanced noise cancel...,2.896764,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",The response is safe as it does not contain an...,1.0,The response directly and clearly provides bot...,5.0
2,Get details for usb charger,"[{'tool_name': 'get_product_details', 'tool_in...",A super fast and light usb charger,2.39513,0,"[{""tool_name"": ""get_product_details"", ""tool_in...","The response is a simple, harmless description...",1.0,The response is highly incoherent with respect...,1.0
3,Get product details and price for shoes,"[{'tool_name': 'get_product_details', 'tool_in...","For shoes, the product details are: High-perfo...",2.680098,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",The response provides product details and a pr...,1.0,"The response has a seamless logical flow, expe...",5.0
4,Get product details for speaker?,"[{'tool_name': 'get_product_details', 'tool_in...",A voice-controlled smart speaker that plays mu...,3.005949,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",The response provides a neutral description of...,1.0,"The response is a single, concise sentence tha...",5.0


#### Visualize evaluation results


Print new evaluation result sample.

In [21]:
display_dataframe_rows(response_eval_result.metrics_table, num_rows=3)

### Evaluate generated response conditioned by tool choosing

When evaluating AI agents that interact with environments, standard text generation metrics like coherence may not be sufficient. This is because these metrics primarily focus on text structure, while agent responses should be assessed based on their effectiveness within the environment.

Instead, use custom metrics that assess whether the agent's response logically follows from its tools choices like the one you have in this section.

#### Define a custom metric

According to the [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval#model-based-metrics), you can define a prompt template for evaluating whether an AI agent's response follows logically from its actions by setting up criteria and a rating system for this evaluation.

Define a `criteria` to set the evaluation guidelines and a `pointwise_rating_rubric` to provide a scoring system (1 or 0). Then use a `PointwiseMetricPromptTemplate` to create the template using these components.


In [22]:
criteria = {
    "Follows trajectory": (
        "Evaluate whether the agent's response logically follows from the "
        "sequence of actions it took. Consider these sub-points:\n"
        "  - Does the response reflect the information gathered during the trajectory?\n"
        "  - Is the response consistent with the goals and constraints of the task?\n"
        "  - Are there any unexpected or illogical jumps in reasoning?\n"
        "Provide specific examples from the trajectory and response to support your evaluation."
    )
}

pointwise_rating_rubric = {
    "1": "Follows trajectory",
    "0": "Does not follow trajectory",
}

response_follows_trajectory_prompt_template = PointwiseMetricPromptTemplate(
    criteria=criteria,
    rating_rubric=pointwise_rating_rubric,
    input_variables=["prompt", "predicted_trajectory"],
)

Print the prompt_data of this template containing the combined criteria and rubric information ready for use in an evaluation.

In [23]:
print(response_follows_trajectory_prompt_template.prompt_data)

# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models. We will provide you with the user prompt and an AI-generated responses.
You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps. Give step by step explanations for your rating, and only choose ratings from the Rating Rubric.


# Evaluation
## Criteria
Follows trajectory: Evaluate whether the agent's response logically follows from the sequence of actions it took. Consider these sub-points:
  - Does the response reflect the information gathered during the trajectory?
  - Is the response consistent with the goals and constraints of the task?
  - Are there any unexpected or illogical jumps in reasoning?
Provide specific examples from the trajectory and response

After you define the evaluation prompt template, set up the associated metric to evaluate how well a response follows a specific trajectory. The `PointwiseMetric` creates a metric where `response_follows_trajectory` is the metric's name and `response_follows_trajectory_prompt_template` provides instructions or context for evaluation you set up before.


In [24]:
response_follows_trajectory_metric = PointwiseMetric(
    metric="response_follows_trajectory",
    metric_prompt_template=response_follows_trajectory_prompt_template,
)

#### Set response metrics

Set new generated response evaluation metrics by including the custom metric.


In [25]:
response_tool_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "safety",
    response_follows_trajectory_metric,
]

#### Run an evaluation task

Run a new agent's evaluation.

In [26]:
EXPERIMENT_RUN = f"response-over-tools-{get_id()}"

response_eval_tool_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=response_tool_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/reasoning-metric-eval",
)

response_eval_tool_result = response_eval_tool_task.evaluate(
    # Uncomment the line below if you are providing the agent with an unparsed dataset
    runnable=agent_parsed_outcome_sync, 
    experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_tool_result)

I0403 02:49:02.697318 2309781 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:02.906130 2309813 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:02.906339 2309813 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:02.962272 2309839 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:02.962372 2309839 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:03.179912 2309871 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:03.180094 2309871 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:03.180098 2309871 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:03.247369 2309899 ev_poll_posix.cc:593] FD from fork parent still in

Associating projects/959676885215/locations/europe-west1/metadataStores/default/contexts/evaluate-adk-agent-response-over-tools-9eowicyp to Experiment: evaluate-adk-agent


I0403 02:49:04.492437 2310140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:04.492654 2310140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:04.492697 2310140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:04.492698 2310140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:04.694790 2310179 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:04.694988 2310179 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:04.695035 2310179 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:04.695037 2310179 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)


Logging Eval experiment evaluation metadata: {'output_file': 'gs://immersion-day-sandbox-008-optimization/evaluation//reasoning-metric-eval/eval_results_2026-04-03-02-49-01-830b3.csv'}


I0403 02:49:04.908771 2310209 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:04.908948 2310209 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:04.908981 2310209 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:04.908982 2310209 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:05.126158 2310239 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:05.126370 2310239 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:05.126422 2310239 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:05.126425 2310239 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
  0%|          | 0/5 [00:00<?, ?it/s]I0403 02:49:05.218772 2310276 ev_poll_posix

...

I0403 02:49:25.931082 2312115 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(108, generation: 1)
I0403 02:49:26.120593 2312140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(108, generation: 1)
I0403 02:49:26.120674 2312140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:26.120676 2312140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:26.120677 2312140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:26.120678 2312140 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)


### Summary Metrics

,metric,value
0,row_count,5.000000
1,trajectory_exact_match/mean,1.000000
2,trajectory_exact_match/std,0.000000
3,trajectory_in_order_match/mean,1.000000
4,trajectory_in_order_match/std,0.000000
5,safety/mean,1.000000
6,safety/std,0.000000
7,response_follows_trajectory/mean,1.000000
8,response_follows_trajectory/std,0.000000
9,latency_in_seconds/mean,3.149700


### Row‑wise Metrics

,prompt,reference_trajectory,response,latency_in_seconds,failure,predicted_trajectory,trajectory_exact_match/score,trajectory_in_order_match/score,safety/explanation,safety/score,response_follows_trajectory/explanation,response_follows_trajectory/score
0,Get price for smartphone,"[{'tool_name': 'get_product_price', 'tool_inpu...",The price for a smartphone is $500.,2.77189,0,"[{""tool_name"": ""get_product_price"", ""tool_inpu...",1.0,1.0,The response is safe as it does not contain an...,1.0,The AI's response 'The price for a smartphone ...,1.0
1,Get product details and price for headphones,"[{'tool_name': 'get_product_details', 'tool_in...",Wireless headphones with advanced noise cancel...,3.745114,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,The response provides product details and a pr...,1.0,The response accurately provides both product ...,1.0
2,Get details for usb charger,"[{'tool_name': 'get_product_details', 'tool_in...",A super fast and light usb charger,2.503353,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,The response is a neutral description of a USB...,1.0,The trajectory indicates a call to 'get_produc...,1.0
3,Get product details and price for shoes,"[{'tool_name': 'get_product_details', 'tool_in...",The product details for shoes are: High-perfor...,3.788797,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,The response is safe as it contains no hate sp...,1.0,The response accurately provides both the prod...,1.0
4,Get product details for speaker?,"[{'tool_name': 'get_product_details', 'tool_in...",A voice-controlled smart speaker that plays mu...,2.939348,0,"[{""tool_name"": ""get_product_details"", ""tool_in...",1.0,1.0,The response is free from any toxic language o...,1.0,The response accurately provides product detai...,1.0


#### Visualize evaluation results

Visualize evaluation result sample.

In [27]:
display_dataframe_rows(response_eval_tool_result.metrics_table, num_rows=3)

In [28]:
plot_bar_plot(
    response_eval_tool_result,
    title="Response Metrics",
    metrics=[f"{metric}/mean" for metric in response_tool_metrics],
)

--- plot_bar_plot ---

📊 --- Metrics Bar Chart ---
exact_match               | ████████████████████ (1.00)
in_order_match            | ████████████████████ (1.00)
safety                    | ████████████████████ (1.00)
response_follows_trajectory | ████████████████████ (1.00)
----------------------------



## Bonus: Bring-Your-Own-Dataset (BYOD) and evaluate a LangGraph agent using Vertex AI Gen AI Evaluation

In Bring Your Own Dataset (BYOD) [scenarios](https://cloud.google.com/vertex-ai/generative-ai/docs/models/evaluation-dataset), you provide both the predicted trajectory and the generated response from the agent.


### Bring your own evaluation dataset

Define the evaluation dataset with the predicted trajectory and the generated response.

In [29]:
byod_eval_data = {
    "prompt": [
        "Get price for smartphone",
        "Get product details and price for headphones",
        "Get details for usb charger",
        "Get product details and price for shoes",
        "Get product details for speaker?",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "smartphone"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "headphones"},
            },
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "headphones"},
            },
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "usb charger"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "shoes"},
            },
            {"tool_name": "get_product_price", "tool_input": {"product_name": "shoes"}},
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "speaker"},
            }
        ],
    ],
    "predicted_trajectory": [
        [
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "smartphone"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "headphones"},
            },
            {
                "tool_name": "get_product_price",
                "tool_input": {"product_name": "headphones"},
            },
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "usb charger"},
            }
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "shoes"},
            },
            {"tool_name": "get_product_price", "tool_input": {"product_name": "shoes"}},
        ],
        [
            {
                "tool_name": "get_product_details",
                "tool_input": {"product_name": "speaker"},
            }
        ],
    ],
    "response": [
        "500",
        "50",
        "A super fast and light usb charger",
        "100",
        "A voice-controlled smart speaker that plays music, sets alarms, and controls smart home devices.",
    ],
}

byod_eval_sample_dataset = pd.DataFrame(byod_eval_data)
byod_eval_sample_dataset["predicted_trajectory"] = byod_eval_sample_dataset[
    "predicted_trajectory"
].apply(json.dumps)
byod_eval_sample_dataset["reference_trajectory"] = byod_eval_sample_dataset[
    "reference_trajectory"
].apply(json.dumps)
byod_eval_sample_dataset["response"] = byod_eval_sample_dataset["response"].apply(json.dumps)

### Run an evaluation task

Run a new agent's evaluation using your own dataset and the same setting of the latest evaluation.

In [30]:
EXPERIMENT_RUN_NAME = f"response-over-tools-byod-{get_id()}"

byod_response_eval_tool_task = EvalTask(
    dataset=byod_eval_sample_dataset,
    metrics=response_tool_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/byod-eval",
)

byod_response_eval_tool_result = byod_response_eval_tool_task.evaluate(
    experiment_run_name=EXPERIMENT_RUN_NAME
)

display_eval_report(byod_response_eval_tool_result)

I0403 02:49:32.479862 2312353 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(108, generation: 1)
I0403 02:49:32.479959 2312353 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:32.479961 2312353 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:32.479962 2312353 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:32.479964 2312353 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:32.682523 2312387 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(108, generation: 1)
I0403 02:49:32.682668 2312387 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:32.682670 2312387 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:32.682671 2312387 ev_poll_posix.cc:593] FD from fork parent still in

I0403 02:49:32.887584 2312418 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(108, generation: 1)
I0403 02:49:32.887756 2312418 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:32.887778 2312418 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:32.887779 2312418 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:32.887780 2312418 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:33.084625 2312452 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(108, generation: 1)
I0403 02:49:33.084855 2312452 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:33.084989 2312452 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:33.084990 2312452 ev_poll_posix.cc:593] FD from fork parent still in

Associating projects/959676885215/locations/europe-west1/metadataStores/default/contexts/evaluate-adk-agent-response-over-tools-byod-ibm4jcnp to Experiment: evaluate-adk-agent


I0403 02:49:35.162878 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)
I0403 02:49:35.163083 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0403 02:49:35.163168 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:49:35.163169 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:49:35.163170 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:35.163171 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:35.163172 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:35.163173 2312866 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:35.364414 2312896 ev_poll_posix.cc:593] FD from fork parent still in p

Logging Eval experiment evaluation metadata: {'output_file': 'gs://immersion-day-sandbox-008-optimization/evaluation//byod-eval/eval_results_2026-04-03-02-49-32-f6747.csv'}


...

I0403 02:49:43.909553 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(105, generation: 1)
I0403 02:49:43.909707 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0403 02:49:43.909772 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(99, generation: 1)
I0403 02:49:43.909811 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(97, generation: 1)
I0403 02:49:43.909812 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(106, generation: 1)
I0403 02:49:43.909814 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
I0403 02:49:43.909815 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(103, generation: 1)
I0403 02:49:43.909816 2313355 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0403 02:49:44.087284 2313385 ev_poll_posix.cc:593] FD from fork parent still in p

### Summary Metrics

,metric,value
0,row_count,5.000000
1,trajectory_exact_match/mean,1.000000
2,trajectory_exact_match/std,0.000000
3,trajectory_in_order_match/mean,1.000000
4,trajectory_in_order_match/std,0.000000
5,safety/mean,1.000000
6,safety/std,0.000000
7,response_follows_trajectory/mean,0.600000
8,response_follows_trajectory/std,0.547723


### Row‑wise Metrics

,prompt,reference_trajectory,predicted_trajectory,response,trajectory_exact_match/score,trajectory_in_order_match/score,safety/explanation,safety/score,response_follows_trajectory/explanation,response_follows_trajectory/score
0,Get price for smartphone,"[{""tool_name"": ""get_product_price"", ""tool_inpu...","[{""tool_name"": ""get_product_price"", ""tool_inpu...","""500""",1.0,1.0,The response '500' is a simple numerical value...,1.0,The trajectory indicates that the 'get_product...,1.0
1,Get product details and price for headphones,"[{""tool_name"": ""get_product_details"", ""tool_in...","[{""tool_name"": ""get_product_details"", ""tool_in...","""50""",1.0,1.0,The response '50' is a numerical output and do...,1.0,The response '50' only provides a potential pr...,0.0
2,Get details for usb charger,"[{""tool_name"": ""get_product_details"", ""tool_in...","[{""tool_name"": ""get_product_details"", ""tool_in...","""A super fast and light usb charger""",1.0,1.0,The response is a simple descriptive phrase ab...,1.0,The trajectory correctly calls the `get_produc...,1.0
3,Get product details and price for shoes,"[{""tool_name"": ""get_product_details"", ""tool_in...","[{""tool_name"": ""get_product_details"", ""tool_in...","""100""",1.0,1.0,The response '100' is a simple numerical outpu...,1.0,The response '100' only provides a numerical v...,0.0
4,Get product details for speaker?,"[{""tool_name"": ""get_product_details"", ""tool_in...","[{""tool_name"": ""get_product_details"", ""tool_in...","""A voice-controlled smart speaker that plays m...",1.0,1.0,"The response is a neutral, factual description...",1.0,The response directly provides product details...,1.0


### Visualize evaluation results

Visualize evaluation result sample.

In [31]:
display_dataframe_rows(byod_response_eval_tool_result.metrics_table, num_rows=3)

In [32]:
display_radar_plot(
    byod_response_eval_tool_result,
    title="ADK agent evaluation",
    metrics=[f"{metric}/mean" for metric in response_tool_metrics],
)

--- display_radar_plot ---

📊 --- Metrics Chart ---
exact_match               | ████████████████████ (1.00)
in_order_match            | ████████████████████ (1.00)
safety                    | ████████████████████ (1.00)
response_follows_trajectory | ████████████ (0.60)
------------------------



## Cleaning up


In [ ]:
delete_experiment = True

if delete_experiment:
    try:
        experiment = aiplatform.Experiment(EXPERIMENT_NAME)
        experiment.delete(delete_backing_tensorboard_runs=True)
    except Exception as e:
        print(e)